# [8.3] ACDC and Circuit Metrics - Exercises

Automated circuit discovery is a proposal mechanism, not a proof. This notebook teaches the metric battery that turns a proposed circuit into something falsifiable: exact patch scores, ACDC-style threshold pruning, faithfulness, minimality, completeness, same-size random baselines, held-out template checks, and exact-vs-approximate method comparison.

The accepted CUDA result is deliberately scoped:

```text
exact patch scores by residual position: [0, 0, 0, 0, 0, 1]
kept circuit: ["position_5"]
faithfulness / minimality / completeness / random / held-out checks: pass
```

A section-ready result is not just green tests. You should be able to explain which claim each metric supports, which failure mode it catches, and why this is a position-circuit preflight rather than a full ACDC IOI replication.

<details>
<summary>Expected output</summary>

By the end of the notebook, your CPU smoke report should contain `num_kept = 2`, faithfulness preserved fraction `0.8`, minimality damage `1.7`, omitted-node gain `0.15`, random-baseline margin `1.4`, OOD worst-template accuracy `1.0`, and exact-vs-approximate method comparison passing for the good method. The committed CUDA report should prune to `position_5` and pass the metric battery.

</details>

<details>
<summary>Help - how this follows original ARENA IOI</summary>

Original ARENA moves from a task metric to causal interventions and then to circuit-level validation. This section extracts the validation part: after a method proposes a circuit, you check whether it preserves behavior, is not bloated, is not missing important omitted nodes, beats same-size random controls, and works beyond the selection prompt.

</details>


In [ ]:
import json
import sys
from collections.abc import Mapping
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t

chapter = "chapter8_automated_circuits"
section = "part3_acdc_circuit_metrics"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_acdc_circuit_metrics.tests as tests
import part3_acdc_circuit_metrics.utils as utils

GT_TIER = "GT-1"
EXERCISE_ID = "8_3_acdc_and_circuit_metrics"
DIFFICULTY = 4
IMPORTANCE = 4
EXPECTED_RUNTIME = "35-50 minutes for exercises; about 1-2 minutes for the CUDA preflight"
REQUIRES_GPU = True

@dataclass(frozen=True)
class ActivationPatchingSweep:
    patch_scores: t.Tensor
    best_index: int
    best_score: float


@dataclass(frozen=True)
class ACDCPruningReport:
    kept_edges: tuple[str, ...]
    removed_edges: tuple[str, ...]
    threshold: float
    num_kept: int


@dataclass(frozen=True)
class CircuitFaithfulnessReport:
    full_metric: float
    corrupt_metric: float
    circuit_metric: float
    preserved_fraction: float
    passes_faithfulness: bool


@dataclass(frozen=True)
class CircuitMinimalityReport:
    circuit_metric: float
    ablated_metric: float
    metric_damage: float
    passes_minimality: bool


@dataclass(frozen=True)
class CircuitCompletenessReport:
    circuit_metric: float
    expanded_metric: float
    omitted_node_gain: float
    passes_completeness: bool


@dataclass(frozen=True)
class RandomCircuitBaselineReport:
    circuit_metric: float
    random_metric: float
    margin: float
    circuit_beats_random: bool


@dataclass(frozen=True)
class OODTemplateReport:
    per_template_accuracy: dict[int, float]
    worst_template_accuracy: float
    passes_ood: bool


@dataclass(frozen=True)
class CircuitMethodComparisonReport:
    exact_top_edges: tuple[str, ...]
    method_top_edges: dict[str, tuple[str, ...]]
    topk_overlap: dict[str, float]
    score_correlations: dict[str, float]
    circuit_sizes: dict[str, int]
    best_matching_method: str
    passes_comparison: bool


## Patch-Recovery Scores

The metric defines the behavioral direction. Here we use positive-minus-negative answer logit difference, then normalize patched metrics between corrupt and clean behavior.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_position_patching_helpers_score_recovery` passed!
All tests in `test_position_patching_helpers_reject_degenerate_inputs` passed!
```

The toy logits should produce logit diff `4.0`; patched metrics `[-1.0, 1.0, 3.0]` should produce recovery scores `[0.0, 0.5, 1.0]`.

</details>

<details>
<summary>Help - why normalize?</summary>

Raw logit differences vary across prompts. Normalized recovery gives a common scale: `0` means corrupt behavior and `1` means clean behavior.

</details>

<details>
<summary>Common bug</summary>

Returning raw patched metrics instead of recovered fractions will make the pruning threshold depend on prompt scale.

</details>

<details>
<summary>Solution sketch</summary>

Validate token ids and finiteness, compute a scalar mean logit diff, reject degenerate clean/corrupt gaps, and return `(patched - corrupt) / (clean - corrupt)` with the best component index.

</details>


In [ ]:
def answer_logit_diff(
    logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> float:
    raise NotImplementedError()


def activation_patching_sweep(
    *,
    clean_metric: float,
    corrupt_metric: float,
    patched_metrics: t.Tensor,
) -> ActivationPatchingSweep:
    raise NotImplementedError()


tests.test_position_patching_helpers_score_recovery(
    answer_logit_diff,
    activation_patching_sweep,
)
tests.test_position_patching_helpers_reject_degenerate_inputs(
    answer_logit_diff,
    activation_patching_sweep,
)


## ACDC-Style Pruning

ACDC-style pruning starts with candidate edges or nodes and removes low-scoring candidates. Keep the candidate names attached to their scores; a circuit report without named components is hard to inspect.

> ```yaml
> Difficulty: easy
> Importance: high
> You should spend 5 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_acdc_pruning_report_keeps_threshold_edges` passed!
All tests in `test_acdc_pruning_report_rejects_bad_scores_or_names` passed!
```

For scores `[0.9, 0.5, 0.2]`, names `['name-mover', 'backup', 'negative']`, and threshold `0.5`, the kept edges should be `('name-mover', 'backup')`.

</details>

<details>
<summary>Help - pruning is only a proposal</summary>

Pruning says which components to test next. It does not prove faithfulness, minimality, completeness, or robustness.

</details>

<details>
<summary>Common bug</summary>

Do not use a strict `>` threshold here. The contract keeps scores equal to the threshold.

</details>

<details>
<summary>Solution sketch</summary>

Flatten scores, reject empty or non-finite scores, require score/name alignment and nonempty names, then split names by `score >= threshold`.

</details>


In [ ]:
def acdc_pruning_report(
    edge_scores: t.Tensor,
    edge_names: list[str],
    *,
    threshold: float,
) -> ACDCPruningReport:
    raise NotImplementedError()


tests.test_acdc_pruning_report_keeps_threshold_edges(acdc_pruning_report)
tests.test_acdc_pruning_report_rejects_bad_scores_or_names(acdc_pruning_report)


## Circuit Metric Battery

Faithfulness asks whether the circuit preserves behavior. Minimality asks whether removing circuit parts hurts. Completeness asks whether adding the best omitted parts helps little. A same-size random baseline catches trivial circuits.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 20 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_faithfulness_report_normalizes_clean_corrupt_gap` passed!
All tests in `test_minimality_and_completeness_reports_distinguish_failure_modes` passed!
All tests in `test_random_circuit_baseline_report_requires_margin` passed!
All tests in `test_circuit_metric_reports_reject_invalid_thresholds` passed!
```

The toy faithful circuit should preserve `0.8` of the gap, minimality damage should be `1.7`, omitted-node gain should be `0.15`, and random-baseline margin should be `1.4`.

</details>

<details>
<summary>Help - separate the failure modes</summary>

A circuit with every component can be faithful but not minimal. A tiny circuit can be minimal but incomplete. Keep separate reports so the failure mode stays visible.

</details>

<details>
<summary>Common bug</summary>

Completeness uses `expanded_metric - circuit_metric`; if this is large, the circuit is missing something important.

</details>

<details>
<summary>Solution sketch</summary>

Compute preserved fraction using the clean-corrupt denominator, minimality damage as `circuit - ablated`, omitted-node gain as `expanded - circuit`, and random margin as `circuit - random`.

</details>


In [ ]:
def circuit_faithfulness_report(
    *,
    full_metric: float,
    corrupt_metric: float,
    circuit_metric: float,
    min_preserved_fraction: float = 0.75,
) -> CircuitFaithfulnessReport:
    raise NotImplementedError()


def circuit_minimality_report(
    *,
    circuit_metric: float,
    ablated_metric: float,
    min_metric_damage: float = 0.5,
) -> CircuitMinimalityReport:
    raise NotImplementedError()


def circuit_completeness_report(
    *,
    circuit_metric: float,
    expanded_metric: float,
    max_omitted_node_gain: float = 0.2,
) -> CircuitCompletenessReport:
    raise NotImplementedError()


def random_circuit_baseline_report(
    *,
    circuit_metric: float,
    random_metric: float,
    min_margin: float = 0.5,
) -> RandomCircuitBaselineReport:
    raise NotImplementedError()


tests.test_faithfulness_report_normalizes_clean_corrupt_gap(circuit_faithfulness_report)
tests.test_minimality_and_completeness_reports_distinguish_failure_modes(
    circuit_minimality_report,
    circuit_completeness_report,
)
tests.test_random_circuit_baseline_report_requires_margin(random_circuit_baseline_report)
tests.test_circuit_metric_reports_reject_invalid_thresholds(
    circuit_faithfulness_report,
    circuit_minimality_report,
    circuit_completeness_report,
    random_circuit_baseline_report,
)


## Held-Out Template Controls

A circuit should not only work on the prompt that selected it. Track per-template accuracy and the worst template.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_ood_template_report_tracks_worst_template` passed!
All tests in `test_ood_template_report_rejects_degenerate_inputs` passed!
```

The failing toy report should expose per-template accuracy `{0: 1.0, 1: 0.5}` and fail because the worst template is below threshold.

</details>

<details>
<summary>Help - why worst-template accuracy?</summary>

Average accuracy can hide a broken prompt family. The worst template is the first place to look for brittleness.

</details>

<details>
<summary>Common bug</summary>

Do not accept answer ids outside the vocabulary dimension just because `argmax` returns an integer.

</details>

<details>
<summary>Solution sketch</summary>

Validate shapes, vocabulary ids, finite logits, and `min_accuracy`; group examples by template id and compute each template's argmax accuracy.

</details>


In [ ]:
def ood_template_report(
    logits: t.Tensor,
    answer_ids: t.Tensor,
    template_ids: t.Tensor,
    *,
    min_accuracy: float = 0.75,
) -> OODTemplateReport:
    raise NotImplementedError()


tests.test_ood_template_report_tracks_worst_template(ood_template_report)
tests.test_ood_template_report_rejects_degenerate_inputs(ood_template_report)


## Exact-vs-Approximate Circuit Comparison

When exact patching is available, it defines the reference top-k circuit. Approximate methods should report both top-k overlap and score correlation against that reference.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_circuit_method_comparison_report_matches_exact_patching` passed!
All tests in `test_circuit_method_comparison_report_rejects_bad_inputs` passed!
```

In the toy example, `eap_ig` should recover the exact top-2 edges while `weak_eap` should recover none and make the comparison fail.

</details>

<details>
<summary>Help - why require every method to pass?</summary>

The report should not let a good method hide a bad one. If a comparison method fails, that failure should be visible.

</details>

<details>
<summary>Common bug</summary>

Do not drop edge names after computing top-k indices; the report should name the candidate circuit.

</details>

<details>
<summary>Solution sketch</summary>

Flatten scores, validate thresholds and shapes, compute exact top-k edge names, compute each method's top-k overlap and Pearson correlation, then pass only if every method clears both thresholds.

</details>


In [ ]:
def circuit_method_comparison_report(
    exact_scores: t.Tensor,
    method_scores: Mapping[str, t.Tensor],
    edge_names: list[str],
    *,
    top_k: int,
    min_topk_overlap: float = 0.5,
    min_score_correlation: float = 0.5,
) -> CircuitMethodComparisonReport:
    raise NotImplementedError()


tests.test_circuit_method_comparison_report_matches_exact_patching(
    circuit_method_comparison_report,
)
tests.test_circuit_method_comparison_report_rejects_bad_inputs(
    circuit_method_comparison_report,
)


## Combined Contract

Compose the local helpers into a CPU-only smoke report before touching the live model.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

The contract should include passing ACDC pruning, faithfulness, minimality, completeness, random-baseline, OOD, and method-comparison reports.

</details>

<details>
<summary>Help - reading the combined contract</summary>

The smoke report proves report wiring, not real circuit discovery. The CUDA report is the real-model mechanics evidence.

</details>

<details>
<summary>Solution sketch</summary>

Use the toy fixtures above, convert every dataclass report to a dictionary, and return one JSON-like dictionary.

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)


## Signature Result

The committed CUDA report is the section-scale result: a pinned `gelu-1l` exact residual-position patching sweep prunes to `position_5`, then the one-position circuit passes the metric battery.

<details>
<summary>Expected output</summary>

```text
preflight_passed: true
patch_scores_by_position: [0, 0, 0, 0, 0, 1]
kept_edges: ["position_5"]
preserved_fraction: 1.0
minimality_metric_damage: 6.2771
omitted_node_gain: 0.0
random_baseline_margin: 6.2771
passes_ood: true
```

</details>

<details>
<summary>Help - what does this result prove?</summary>

It validates the local circuit-metric harness on a real model path. It does not prove full ACDC, IOI circuit discovery, or a head/feature/path circuit.

</details>

<details>
<summary>Common bug</summary>

Do not call final-position localization a discovered mechanistic circuit. It is a scoped metrics preflight.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["preflight_passed"]
    assert gpu["patch_scores_by_position"] == [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
    assert gpu["best_position"] == gpu["target_position"] == 5
    assert gpu["kept_edges"] == ["position_5"]
    assert gpu["passes_faithfulness"]
    assert gpu["passes_minimality"]
    assert gpu["passes_completeness"]
    assert gpu["circuit_beats_random"]
    assert gpu["passes_ood"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
utils.print_report(
    "Committed CUDA ACDC/circuit-metric report",
    {
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "device": gpu["device"],
        "patch_scores": gpu["patch_scores_by_position"],
        "kept_edges": gpu["kept_edges"],
        "preserved_fraction": gpu["preserved_fraction"],
        "minimality_damage": round(gpu["minimality_metric_damage"], 4),
        "omitted_node_gain": gpu["omitted_node_gain"],
        "random_margin": round(gpu["random_baseline_margin"], 4),
        "template_recoveries": gpu["template_recoveries"],
        "peak_vram_gb": round(gpu["peak_vram_gb"], 4),
    },
)


## Limitations

This is a GT-1 position-circuit metric preflight on one pinned `gelu-1l` hook, one primary prompt pair, and three held-out safe prompt-template pairs. It is not full ACDC, not IOI replication, not greater-than circuit discovery, and not evidence for a layer/head/feature/path circuit.

## Further Research

Implement real edge-level ACDC, compare thresholded exact-patch circuits to EAP/EAP-IG across several granularities, add threshold-sensitivity curves, and reproduce a small IOI or greater-than fragment with real path-patching semantics.
